# Digital Twin Extraction powered by CASCADES

This notebook runs the CASCADES Server (Qwen3-4B with NF4 quantization and CASCADES adapters) and processes your personal Google Takeout data chunks into Neo4j Cypher statements, while simultaneously feeding the structured facts into Hemisphere B for parametric memory consolidation.

It is designed to run on a **T4 GPU** (free tier Google Colab).

## Setup Environment

In [ ]:
!pip install torch torchvision torchaudio
!pip install transformers accelerate bitsandbytes tokenizers
!pip install fastapi uvicorn sse-starlette requests

## Upload Google Takeout Data & Model

1. Mount your Google Drive
2. Ensure `abliteratedqwen3.zip` is in the root of your Google Drive
3. Put your `takeout_chunks_32k` data in your drive
4. Put `CASCADES_Colab_Minimal.zip` in your drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# We will extract the model directly to the fast local Colab NVMe disk
MODEL_PATH = "/content/abliteratedqwen3"
CHUNKS_DIR = "/content/drive/MyDrive/takeout_chunks_32k"
CYPHER_DIR = "/content/drive/MyDrive/cypher_output"

import os
os.makedirs(CYPHER_DIR, exist_ok=True)

if not os.path.exists(MODEL_PATH):
    print("Unzipping model to local colab storage (this will take a minute but makes inference much faster)...")
    !unzip -q /content/drive/MyDrive/abliteratedqwen3.zip -d /content/
    print("Model unzipped!")

## Install the CASCADES App

In [ ]:
!mkdir -p /content/CASCADES_REPO
!cp /content/drive/MyDrive/CASCADES_Colab_Minimal.zip /content/CASCADES_REPO/
%cd /content/CASCADES_REPO/
!unzip -o -q CASCADES_Colab_Minimal.zip

import sys
REPO_DIR = "/content/CASCADES_REPO"
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

## Start CASCADES Server in Background

In [ ]:
import subprocess
import time
import requests

# Start the server using the model on Drive
# Using subprocess so we don't block the notebook
print(f"Starting CASCADES Server from {MODEL_PATH}...")

cmd = [
    "python", "-m", "app.server",
    "--model_id", MODEL_PATH
]

server_process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE, 
    stderr=subprocess.STDOUT,
    text=True
)

# Wait for server to come up
CASCADES_URL = "http://127.0.0.1:8000"

print("Waiting for model to load (may take 2-3 minutes)...")
started = False
for i in range(60):
    try:
        resp = requests.get(f"{CASCADES_URL}/v1/memory/stats", timeout=2)
        if resp.status_code == 200:
            print("\nServer is UP!")
            started = True
            break
    except:
        print(".", end="", flush=True)
        time.sleep(5)

if not started:
    print("\nServer failed to start. Check logs:")
    for i in range(20):
        print(server_process.stdout.readline())
    raise Exception("Server didn't start")

## Run The Extraction Pipeline

In [ ]:
# Overwrite the script's directories to use the specific paths
import re

script_path = os.path.join(REPO_DIR, 'local_extract_cascades.py')
if os.path.exists(script_path):
    with open(script_path, 'r', encoding='utf-8') as f:
        content = f.read()
        
    # Set new paths
    content = re.sub(
        r'CHUNKS_DIR = r".*?"',
        f'CHUNKS_DIR = r"{CHUNKS_DIR}"',
        content
    )
    content = re.sub(
        r'CYPHER_DIR = r".*?"',
        f'CYPHER_DIR = r"{CYPHER_DIR}"',
        content
    )
    
    with open(script_path, 'w', encoding='utf-8') as f:
        f.write(content)
    print("Updated directories in local_extract_cascades.py")
else:
    print(f"Could not find {script_path}")

In [ ]:
!python local_extract_cascades.py

## Manually Check Memory Consolidation Stats

In [ ]:
import requests
resp = requests.get("http://127.0.0.1:8000/v1/memory/stats")
print(resp.json())

## Save Learned Weights to Drive
After processing, the adapter has absorbed the knowledge into parametric memory.

In [ ]:
# The state is maintained in RAM by the server.
# To save it to a .pt file on your Google Drive:
import requests
import shutil
import os
from google.colab import files

print("Ensure you have exported the memory weights!")
# Example of making an imaginary save endpoint request if we added it to server.py:
# requests.post('http://127.0.0.1:8000/v1/memory/save', json={'path':'/content/drive/MyDrive/my_digital_twin_weights.pt'})